# Lesson 22: Time Series Forecasting with ARIMA & SARIMAX

Scale hourly electricity load with `MinMaxScaler`, fit `statsmodels.tsa.statespace.sarimax.SARIMAX`, run walk-forward validation, and compute one-step and multi-step MAPE.

In [ ]:
!pip install -q statsmodels scikit-learn pandas

In [ ]:
import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.statespace.sarimax import SARIMAX
warnings.filterwarnings('ignore')

rng = np.random.default_rng(22)
idx = pd.date_range('2014-11-01', '2014-12-31 23:00:00', freq='h')
load = 3100 + 500 * np.sin(2 * np.pi * (idx.hour - 7) / 24.0) + rng.normal(0, 40, len(idx))
energy = pd.DataFrame({'load': load}, index=idx)

train = energy.loc[:'2014-12-29 23:00:00'].copy()
test = energy.loc['2014-12-30 00:00:00':].copy()

scaler = MinMaxScaler()
train['load'] = scaler.fit_transform(train[['load']])
test['load'] = scaler.transform(test[['load']])

history = list(train['load'].iloc[-360:])
preds, actuals = [], []
for t in range(12):
    model = SARIMAX(history, order=(2, 1, 0), seasonal_order=(1, 1, 0, 24))
    yhat = model.fit(disp=False).forecast(steps=1)[0]
    preds.append(yhat)
    obs = test['load'].iloc[t]
    actuals.append(obs)
    history.append(obs)
    history.pop(0)

p_mw = scaler.inverse_transform(np.array(preds).reshape(-1, 1)).ravel()
a_mw = scaler.inverse_transform(np.array(actuals).reshape(-1, 1)).ravel()
mape_val = np.mean(np.abs(a_mw - p_mw) / a_mw) * 100
print(f"Walk-Forward 1-step MAPE: {mape_val:.2f}%")

## Assignment: A New ARIMA Model

Increase `HORIZON = 3` and compare `(2, 1, 0)` against `(4, 1, 0)` on both `t+1` and `t+3` MAPE.